# Cicero planner capture mechanistic positive control

This notebook belongs to the project's sequential measurement and validation programme. Read its result as evidence about behavioural validity, representation, comparator strength, timing, information matching, or mechanistic calibration as appropriate. Legacy H1/H_priv identifiers may remain inside code, saved paths, or frozen schemas for reproducibility; they are not the object being "found" by the current analysis.

**Repository framing.** The current paper separates target-specific representation from predictive privilege. A positive neural result is interpreted only after behavioural validity, control, comparator-strength, timing, and information-set checks.


# Notebook 15 — CICERO native planner capture

This notebook instruments the released full CICERO agent at runtime and captures a completed native planner result before outward behavior.

It is an engineering/mechanistic positive-control stage. It does **not** estimate the activation-vs-text H1 effect and does not make a steering claim.

Primary outputs are written under a new run-specific directory in `notebook_outputs/15_cicero_planner_capture_mechanistic_positive_control/`.


## Stage objectives

- Reuse the validated Python 3.7 / PyTorch 1.7.1 / Postman / `pydipcc` stack from Notebook 14.
- Instrument `BQRE1PAgent.run_search` without editing the frozen CICERO checkout.
- Serialize the completed search-result object, zero-argument policy accessors, relevant state/belief fields, and call inputs.
- Capture the next outward message or submitted-order result when one follows the planner event.
- Stop the launched comparison process after the capture target is reached.
- Freeze raw event files, normalized capture artifacts, runtime signatures, and a readiness summary for the following measurement stage.


In [1]:
from __future__ import annotations

import json
import os
import queue
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

from tqdm.auto import tqdm


In [2]:
PROJECT_ROOT = Path("/workspace/latent-reservations")
CICERO_REPO = PROJECT_ROOT / "vendor" / "diplomacy_cicero"

CICERO_COMMIT = "e85afeddb34f5b7c1ea0827203b425a0f7e68ead"

LEGACY_ENV = (
    PROJECT_ROOT
    / ".venvs"
    / "cicero_full_agent_py37_source_v5"
)

LEGACY_PYTHON = (
    LEGACY_ENV
    / "bin"
    / "python"
)

PYTHON_BASE = (
    PROJECT_ROOT
    / "vendor"
    / "cicero_runtime_v5"
    / "cpython-3.7.17"
)

OPENSSL_PREFIX = (
    PROJECT_ROOT
    / "vendor"
    / "cicero_runtime_v5"
    / "openssl-1.1.1w"
)

COMPARE_CONFIG = (
    CICERO_REPO
    / "conf"
    / "c01_ag_cmp"
    / "cmp.prototxt"
)

CICERO_AGENT_INCLUDE = "agents/cicero.prototxt"

NOTEBOOK_BUILD = (
    "latent-reservations-notebook15-cicero-planner-capture-v2"
)

NOTEBOOK_SLUG = (
    "15_cicero_planner_capture_mechanistic_positive_control"
)

RUN_ID = datetime.now(
    timezone.utc
).strftime(
    "%Y%m%dT%H%M%SZ"
)

RUN_DIR = (
    PROJECT_ROOT
    / "notebook_outputs"
    / NOTEBOOK_SLUG
    / RUN_ID
)

MANIFEST_DIR = (
    RUN_DIR
    / "manifests"
)

DATA_DIR = (
    RUN_DIR
    / "data"
)

RUNTIME_DIR = (
    DATA_DIR
    / "runtime"
)

CAPTURE_DIR = (
    DATA_DIR
    / "planner_capture_events"
)

INSTRUMENT_DIR = (
    DATA_DIR
    / "instrumentation"
)

RESULT_DIR = (
    RUN_DIR
    / "results"
)

TABLE_DIR = (
    RESULT_DIR
    / "tables"
)

for path in [
    MANIFEST_DIR,
    RUNTIME_DIR,
    CAPTURE_DIR,
    INSTRUMENT_DIR,
    TABLE_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

CAPTURE_TIMEOUT_SECONDS = 3600

print({
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "run_dir": str(
        RUN_DIR
    ),
})


{'notebook_build': 'latent-reservations-notebook15-cicero-planner-capture-v2', 'run_id': '20260815T140016Z', 'run_dir': '/workspace/latent-reservations/notebook_outputs/15_cicero_planner_capture_mechanistic_positive_control/20260815T140016Z'}


In [3]:
def capture_command(
    command,
    *,
    cwd=None,
    env=None,
    timeout=None,
):
    completed = subprocess.run(
        [
            str(
                item
            )
            for item in command
        ],
        cwd=(
            str(
                cwd
            )
            if cwd is not None
            else None
        ),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=timeout,
    )

    return {
        "returncode": completed.returncode,
        "output": completed.stdout,
    }


def load_capture_events():
    events = []

    for path in sorted(
        CAPTURE_DIR.glob(
            "*.jsonl"
        )
    ):
        with path.open(
            "r",
            encoding="utf-8",
            errors="replace",
        ) as handle:
            for line_number, line in enumerate(
                handle,
                start=1,
            ):
                line = line.strip()

                if not line:
                    continue

                try:
                    event = json.loads(
                        line
                    )
                except json.JSONDecodeError:
                    continue

                event[
                    "_capture_file"
                ] = str(
                    path
                )

                event[
                    "_capture_line"
                ] = line_number

                events.append(
                    event
                )

    events.sort(
        key=lambda row: (
            float(
                row.get(
                    "time_unix",
                    0.0,
                )
            ),
            int(
                row.get(
                    "pid",
                    0,
                )
            ),
            int(
                row.get(
                    "_capture_line",
                    0,
                )
            ),
        )
    )

    return events


def find_capture_completion(
    events,
):
    planner_events = [
        event
        for event in events
        if event.get(
            "event"
        )
        in {
            "run_search_result",
            "run_bilateral_search_with_conditional_evs_result",
        }
    ]

    if not planner_events:
        return {
            "complete": False,
            "planner_event": None,
            "outward_event": None,
        }

    planner_event = planner_events[
        0
    ]

    planner_time = float(
        planner_event.get(
            "time_unix",
            0.0,
        )
    )

    outward_events = [
        event
        for event in events
        if (
            event.get(
                "event"
            )
            in {
                "generate_message_result",
                "get_orders_result",
            }
            and event.get(
                "pid"
            )
            == planner_event.get(
                "pid"
            )
            and float(
                event.get(
                    "time_unix",
                    0.0,
                )
            )
            >= planner_time
        )
    ]

    outward_event = (
        outward_events[
            0
        ]
        if outward_events
        else None
    )

    return {
        "complete": outward_event
        is not None,
        "planner_event": planner_event,
        "outward_event": outward_event,
    }


In [4]:
required_paths = [
    CICERO_REPO,
    LEGACY_PYTHON,
    PYTHON_BASE,
    OPENSSL_PREFIX,
    COMPARE_CONFIG,
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing Notebook 14 runtime prerequisites:\n"
        + "\n".join(
            missing_paths
        )
    )

actual_commit = capture_command(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=CICERO_REPO,
)

if actual_commit[
    "returncode"
] != 0:
    raise RuntimeError(
        actual_commit[
            "output"
        ]
    )

actual_commit_value = actual_commit[
    "output"
].strip()

if actual_commit_value != CICERO_COMMIT:
    raise RuntimeError(
        "Frozen CICERO commit mismatch: "
        + actual_commit_value
    )

stack_markers = sorted(
    LEGACY_ENV.glob(
        ".latent_reservations_cicero_stack_v5_*"
    )
)

if not stack_markers:
    raise RuntimeError(
        "Notebook 14 stack marker is missing."
    )

torch_route_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,pathlib,torch; "
            "root=pathlib.Path(torch.__file__).resolve().parent; "
            "print(json.dumps({"
            "'version':torch.__version__,"
            "'cuda':torch.version.cuda,"
            "'cuda_available':torch.cuda.is_available(),"
            "'root':str(root),"
            "'lib':str(root/'lib')"
            "}))"
        ),
    ],
    cwd=CICERO_REPO,
)

if torch_route_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        torch_route_probe[
            "output"
        ]
    )

torch_route = json.loads(
    torch_route_probe[
        "output"
    ].strip().splitlines()[
        -1
    ]
)

if not torch_route[
    "version"
].startswith(
    "1.7.1"
):
    raise RuntimeError(
        "Expected PyTorch 1.7.1 in the isolated runtime: "
        + json.dumps(
            torch_route,
            sort_keys=True,
        )
    )

TORCH_LIB_DIR = Path(
    torch_route[
        "lib"
    ]
)

legacy_runtime_env = os.environ.copy()

legacy_runtime_env.pop(
    "PYTHONHOME",
    None,
)

legacy_runtime_env.pop(
    "PYTHONPATH",
    None,
)

legacy_runtime_env[
    "PYTHONNOUSERSITE"
] = "1"

legacy_runtime_env[
    "VIRTUAL_ENV"
] = str(
    LEGACY_ENV
)

legacy_runtime_env[
    "PATH"
] = os.pathsep.join(
    [
        str(
            LEGACY_ENV
            / "bin"
        ),
        os.environ.get(
            "PATH",
            "",
        ),
    ]
)

runtime_library_dirs = [
    TORCH_LIB_DIR,
    LEGACY_ENV
    / "lib",
    PYTHON_BASE
    / "lib",
]

for candidate in [
    OPENSSL_PREFIX
    / "lib",
    OPENSSL_PREFIX
    / "lib64",
]:
    if candidate.exists():
        runtime_library_dirs.append(
            candidate
        )

runtime_library_dirs.append(
    Path(
        "/usr/local/cuda/lib64"
    )
)

legacy_runtime_env[
    "LD_LIBRARY_PATH"
] = os.pathsep.join(
    [
        str(
            path
        )
        for path in runtime_library_dirs
        if path.exists()
    ]
)

preflight_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,sys,torch,postman; "
            "from fairdiplomacy import pydipcc; "
            "from fairdiplomacy.agents.bqre1p_agent import BQRE1PAgent; "
            "print(json.dumps({"
            "'python':sys.executable,"
            "'torch':torch.__version__,"
            "'cuda_build':torch.version.cuda,"
            "'cuda_available':torch.cuda.is_available(),"
            "'postman':postman.__file__,"
            "'pydipcc':pydipcc.__file__,"
            "'agent':str(BQRE1PAgent)"
            "}))"
        ),
    ],
    cwd=CICERO_REPO,
    env=legacy_runtime_env,
)

(
    RUNTIME_DIR
    / "preflight_probe.txt"
).write_text(
    preflight_probe[
        "output"
    ]
)

print(
    preflight_probe[
        "output"
    ]
)

if preflight_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Notebook 14 runtime stack did not pass the Notebook 15 preflight."
    )

preflight = {
    "cicero_commit": actual_commit_value,
    "stack_markers": [
        str(
            path
        )
        for path in stack_markers
    ],
    "torch_route": torch_route,
    "runtime_probe": json.loads(
        preflight_probe[
            "output"
        ].strip().splitlines()[
            -1
        ]
    ),
}

(
    TABLE_DIR
    / "preflight.json"
).write_text(
    json.dumps(
        preflight,
        indent=2,
    )
)


{"python": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python", "torch": "1.7.1+cu110", "cuda_build": "11.0", "cuda_available": true, "postman": "/workspace/latent-reservations/vendor/diplomacy_cicero/thirdparty/github/fairinternal/postman/postman/python/postman/__init__.py", "pydipcc": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/pydipcc.cpython-37m-x86_64-linux-gnu.so", "agent": "<class 'fairdiplomacy.agents.bqre1p_agent.BQRE1PAgent'>"}



1103

In [5]:
signature_probe_code = r'''
import inspect
import json

from fairdiplomacy.agents.bqre1p_agent import (
    BQRE1PAgent,
    BRMResult,
)

objects = {
    "BQRE1PAgent.run_search": BQRE1PAgent.run_search,
    "BQRE1PAgent.run_bilateral_search_with_conditional_evs": (
        BQRE1PAgent.run_bilateral_search_with_conditional_evs
    ),
    "BQRE1PAgent.generate_message": BQRE1PAgent.generate_message,
    "BQRE1PAgent.get_orders": BQRE1PAgent.get_orders,
    "BRMResult.get_agent_policy": BRMResult.get_agent_policy,
    "BRMResult.get_bp_policy": BRMResult.get_bp_policy,
    "BRMResult.get_population_policy": BRMResult.get_population_policy,
    "BRMResult.avg_utility": BRMResult.avg_utility,
    "BRMResult.avg_action_utility": BRMResult.avg_action_utility,
}

payload = {}

for name, obj in objects.items():
    try:
        signature = str(
            inspect.signature(
                obj
            )
        )
    except Exception as exc:
        signature = (
            "<signature unavailable: "
            + repr(
                exc
            )
            + ">"
        )

    payload[
        name
    ] = signature

print(
    json.dumps(
        payload,
        indent=2,
        sort_keys=True,
    )
)
'''

signature_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        signature_probe_code,
    ],
    cwd=CICERO_REPO,
    env=legacy_runtime_env,
)

if signature_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        signature_probe[
            "output"
        ]
    )

runtime_signatures = json.loads(
    signature_probe[
        "output"
    ]
)

(
    TABLE_DIR
    / "runtime_signatures.json"
).write_text(
    json.dumps(
        runtime_signatures,
        indent=2,
        sort_keys=True,
    )
)

print(
    json.dumps(
        runtime_signatures,
        indent=2,
        sort_keys=True,
    )
)


{
  "BQRE1PAgent.generate_message": "(self, game: fairdiplomacy.pydipcc.Game, power: str, timestamp: Union[fairdiplomacy.timestamp.Timestamp, NoneType], state: fairdiplomacy.agents.base_agent.AgentState, timings: Union[fairdiplomacy.utils.timing_ctx.TimingCtx, NoneType] = None, recipient: Union[str, NoneType] = None, pseudo_orders: Union[fairdiplomacy.pseudo_orders.PseudoOrders, NoneType] = None) -> Union[fairdiplomacy.typedefs.MessageDict, NoneType]",
  "BQRE1PAgent.get_orders": "(self, game: fairdiplomacy.pydipcc.Game, power: str, state: fairdiplomacy.agents.base_agent.AgentState) -> Tuple[str, ...]",
  "BQRE1PAgent.run_bilateral_search_with_conditional_evs": "(self, game: fairdiplomacy.pydipcc.Game, *, bp_policy: Dict[str, Dict[Tuple[str, ...], float]], early_exit_for_power: Union[str, NoneType] = None, timings: Union[fairdiplomacy.utils.timing_ctx.TimingCtx, NoneType] = None, extra_plausible_orders: Union[Dict[str, List[Tuple[str, ...]]], NoneType] = None, agent_power: str, other_p

In [6]:
SITE_CUSTOMIZE = (
    INSTRUMENT_DIR
    / "sitecustomize.py"
)

INSTRUMENTATION_SOURCE = 'import inspect\nimport json\nimport os\nimport time\nimport traceback\nfrom pathlib import Path\n\n_CAPTURE_DIR_TEXT = os.environ.get(\n    "LR_CICERO_CAPTURE_DIR",\n    "",\n).strip()\n\nif _CAPTURE_DIR_TEXT:\n    import numpy as np\n    import torch\n\n    from fairdiplomacy.agents.bqre1p_agent import (\n        BQRE1PAgent,\n    )\n\n    _CAPTURE_DIR = Path(\n        _CAPTURE_DIR_TEXT\n    )\n    _CAPTURE_DIR.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n    _CAPTURE_PATH = (\n        _CAPTURE_DIR\n        / f"{os.getpid()}.jsonl"\n    )\n\n    _MAX_SEQUENCE = 100\n    _MAX_TENSOR_ELEMENTS = 2048\n    _MAX_DEPTH = 5\n    _MAX_ACTIONS_PER_POWER = 12\n\n    def _json_key(value):\n        if isinstance(value, str):\n            return value\n        if isinstance(value, tuple):\n            return " | ".join(str(item) for item in value)\n        return repr(value)\n\n    def _safe(value, depth=0):\n        if depth > _MAX_DEPTH:\n            return {\n                "__summary__": repr(value)[:500],\n                "__type__": (\n                    type(value).__module__\n                    + "."\n                    + type(value).__name__\n                ),\n            }\n\n        if value is None or isinstance(\n            value,\n            (bool, int, float, str),\n        ):\n            return value\n\n        if isinstance(value, Path):\n            return str(value)\n\n        if isinstance(value, torch.Tensor):\n            tensor = value.detach().cpu()\n            payload = {\n                "__type__": "torch.Tensor",\n                "shape": list(tensor.shape),\n                "dtype": str(tensor.dtype),\n                "numel": int(tensor.numel()),\n            }\n            if tensor.numel() <= _MAX_TENSOR_ELEMENTS:\n                payload["values"] = tensor.tolist()\n            return payload\n\n        if isinstance(value, np.ndarray):\n            payload = {\n                "__type__": "numpy.ndarray",\n                "shape": list(value.shape),\n                "dtype": str(value.dtype),\n                "size": int(value.size),\n            }\n            if value.size <= _MAX_TENSOR_ELEMENTS:\n                payload["values"] = value.tolist()\n            return payload\n\n        if isinstance(value, dict):\n            items = list(value.items())\n            payload = {\n                _json_key(key): _safe(item, depth + 1)\n                for key, item in items[:_MAX_SEQUENCE]\n            }\n            if len(items) > _MAX_SEQUENCE:\n                payload["__truncated_items__"] = len(items) - _MAX_SEQUENCE\n            return payload\n\n        if isinstance(value, (list, tuple, set)):\n            sequence = list(value)\n            payload = [\n                _safe(item, depth + 1)\n                for item in sequence[:_MAX_SEQUENCE]\n            ]\n            if len(sequence) > _MAX_SEQUENCE:\n                payload.append({\n                    "__truncated_items__": len(sequence) - _MAX_SEQUENCE\n                })\n            return payload\n\n        module_name = type(value).__module__\n        class_name = type(value).__name__\n\n        if "pydipcc" in module_name and class_name == "Game":\n            payload = {\n                "__type__": module_name + "." + class_name,\n            }\n            for name in [\n                "current_short_phase",\n                "current_year",\n                "get_state",\n            ]:\n                if not hasattr(value, name):\n                    continue\n                try:\n                    candidate = getattr(value, name)\n                    if callable(candidate):\n                        candidate = candidate()\n                    payload[name] = _safe(candidate, depth + 1)\n                except Exception as exc:\n                    payload[name] = {"__error__": repr(exc)}\n            return payload\n\n        if hasattr(value, "__dict__"):\n            interesting_tokens = (\n                "belief",\n                "policy",\n                "plausible",\n                "action",\n                "utility",\n                "value",\n                "ptype",\n                "player_type",\n                "type_spec",\n                "state",\n                "search_result",\n                "pseudo",\n            )\n            selected = {}\n            for name, item in value.__dict__.items():\n                lower_name = name.lower()\n                if not any(token in lower_name for token in interesting_tokens):\n                    continue\n                selected[name] = _safe(item, depth + 1)\n            return {\n                "__type__": module_name + "." + class_name,\n                "__repr__": repr(value)[:1000],\n                "__selected_state__": selected,\n            }\n\n        return {\n            "__type__": module_name + "." + class_name,\n            "__repr__": repr(value)[:1000],\n        }\n\n    def _emit(event, payload):\n        record = {\n            "event": event,\n            "time_unix": time.time(),\n            "pid": os.getpid(),\n            "ppid": os.getppid(),\n            "payload": _safe(payload),\n        }\n        encoded = json.dumps(\n            record,\n            separators=(",", ":"),\n            sort_keys=True,\n        )\n        with _CAPTURE_PATH.open("a", encoding="utf-8") as handle:\n            handle.write(encoded)\n            handle.write("\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n\n    def _bound_inputs(original, instance, args, kwargs):\n        payload = {}\n        try:\n            signature = inspect.signature(original)\n            bound = signature.bind_partial(\n                instance,\n                *args,\n                **kwargs,\n            )\n            for name, value in bound.arguments.items():\n                if name == "self":\n                    continue\n                if name == "conditional_evs":\n                    try:\n                        size = len(value)\n                    except Exception:\n                        size = None\n                    payload[name] = {\n                        "__type__": (\n                            type(value).__module__\n                            + "."\n                            + type(value).__name__\n                        ),\n                        "__len__": size,\n                        "__content_omitted__": True,\n                    }\n                else:\n                    payload[name] = _safe(value)\n        except Exception as exc:\n            payload = {\n                "__binding_error__": repr(exc),\n                "args": _safe(args),\n                "kwargs": _safe(kwargs),\n            }\n        return payload\n\n    def _zero_arg_method_outputs(result):\n        outputs = {}\n        for name in [\n            "get_agent_policy",\n            "get_bp_policy",\n            "get_population_policy",\n            "is_early_exit",\n        ]:\n            method = getattr(result, name, None)\n            if method is None or not callable(method):\n                continue\n            try:\n                signature = inspect.signature(method)\n                required = [\n                    parameter\n                    for parameter in signature.parameters.values()\n                    if (\n                        parameter.default is inspect.Signature.empty\n                        and parameter.kind\n                        in {\n                            inspect.Parameter.POSITIONAL_ONLY,\n                            inspect.Parameter.POSITIONAL_OR_KEYWORD,\n                            inspect.Parameter.KEYWORD_ONLY,\n                        }\n                    )\n                ]\n                if required:\n                    outputs[name] = {\n                        "__signature__": str(signature),\n                        "__not_called__": True,\n                    }\n                    continue\n                outputs[name] = _safe(method())\n            except Exception as exc:\n                outputs[name] = {\n                    "__error__": repr(exc),\n                    "__traceback__": traceback.format_exc(),\n                }\n        return outputs\n\n    def _utility_outputs(result):\n        payload = {\n            "avg_utility_by_power": {},\n            "top_action_utility_by_power": {},\n        }\n        get_agent_policy = getattr(result, "get_agent_policy", None)\n        avg_utility = getattr(result, "avg_utility", None)\n        avg_action_utility = getattr(result, "avg_action_utility", None)\n        if get_agent_policy is None or not callable(get_agent_policy):\n            return payload\n        try:\n            policy = get_agent_policy()\n        except Exception as exc:\n            payload["__policy_error__"] = repr(exc)\n            return payload\n\n        for power, action_policy in policy.items():\n            power_name = str(power)\n            if avg_utility is not None and callable(avg_utility):\n                try:\n                    payload["avg_utility_by_power"][power_name] = float(\n                        avg_utility(power)\n                    )\n                except Exception as exc:\n                    payload["avg_utility_by_power"][power_name] = {\n                        "__error__": repr(exc)\n                    }\n\n            if (\n                avg_action_utility is None\n                or not callable(avg_action_utility)\n                or not isinstance(action_policy, dict)\n            ):\n                continue\n\n            rows = []\n            ranked = sorted(\n                action_policy.items(),\n                key=lambda item: -float(item[1]),\n            )[:_MAX_ACTIONS_PER_POWER]\n            for action, probability in ranked:\n                row = {\n                    "action": _safe(action),\n                    "policy_probability": float(probability),\n                }\n                try:\n                    row["avg_action_utility"] = float(\n                        avg_action_utility(power, action)\n                    )\n                except Exception as exc:\n                    row["avg_action_utility"] = {\n                        "__error__": repr(exc)\n                    }\n                rows.append(row)\n            payload["top_action_utility_by_power"][power_name] = rows\n        return payload\n\n    def _result_payload(result):\n        return {\n            "type": (\n                type(result).__module__\n                + "."\n                + type(result).__name__\n            ),\n            "repr": repr(result)[:1000],\n            "method_outputs": _zero_arg_method_outputs(result),\n            "utility_outputs": _utility_outputs(result),\n            "beliefs": _safe(getattr(result, "beliefs", None)),\n            "selected_state": _safe(result),\n        }\n\n    def _patch_planner_method(method_name):\n        original = getattr(BQRE1PAgent, method_name, None)\n        if original is None or not callable(original):\n            _emit(\n                "instrumentation_method_missing",\n                {"method": method_name},\n            )\n            return\n\n        def wrapper(self, *args, **kwargs):\n            inputs = _bound_inputs(\n                original,\n                self,\n                args,\n                kwargs,\n            )\n            _emit(\n                method_name + "_begin",\n                {\n                    "planner_method": method_name,\n                    "inputs": inputs,\n                },\n            )\n            started = time.monotonic()\n            try:\n                result = original(\n                    self,\n                    *args,\n                    **kwargs,\n                )\n            except BaseException as exc:\n                _emit(\n                    method_name + "_error",\n                    {\n                        "planner_method": method_name,\n                        "elapsed_seconds": time.monotonic() - started,\n                        "error": repr(exc),\n                        "traceback": traceback.format_exc(),\n                    },\n                )\n                raise\n\n            _emit(\n                method_name + "_result",\n                {\n                    "planner_method": method_name,\n                    "elapsed_seconds": time.monotonic() - started,\n                    "inputs": inputs,\n                    "result": _result_payload(result),\n                },\n            )\n            return result\n\n        wrapper.__name__ = "_lr_capture_" + method_name\n        setattr(BQRE1PAgent, method_name, wrapper)\n\n    _patch_planner_method("run_search")\n    _patch_planner_method("run_bilateral_search_with_conditional_evs")\n\n    def _patch_outward_method(name):\n        original = getattr(BQRE1PAgent, name, None)\n        if original is None or not callable(original):\n            return\n\n        def wrapper(self, *args, **kwargs):\n            inputs = _bound_inputs(\n                original,\n                self,\n                args,\n                kwargs,\n            )\n            started = time.monotonic()\n            try:\n                result = original(\n                    self,\n                    *args,\n                    **kwargs,\n                )\n            except BaseException as exc:\n                _emit(\n                    name + "_error",\n                    {\n                        "elapsed_seconds": time.monotonic() - started,\n                        "error": repr(exc),\n                        "traceback": traceback.format_exc(),\n                        "inputs": inputs,\n                    },\n                )\n                raise\n\n            _emit(\n                name + "_result",\n                {\n                    "elapsed_seconds": time.monotonic() - started,\n                    "inputs": inputs,\n                    "output": _safe(result),\n                },\n            )\n            return result\n\n        wrapper.__name__ = "_lr_capture_" + name\n        setattr(BQRE1PAgent, name, wrapper)\n\n    _patch_outward_method("generate_message")\n    _patch_outward_method("get_orders")\n\n    _emit(\n        "instrumentation_loaded",\n        {\n            "agent_class": str(BQRE1PAgent),\n            "patched_planner_methods": [\n                "run_search",\n                "run_bilateral_search_with_conditional_evs",\n            ],\n        },\n    )\n'

SITE_CUSTOMIZE.write_text(
    INSTRUMENTATION_SOURCE,
    encoding="utf-8",
)

print(
    SITE_CUSTOMIZE
)


/workspace/latent-reservations/notebook_outputs/15_cicero_planner_capture_mechanistic_positive_control/20260815T140016Z/data/instrumentation/sitecustomize.py


In [7]:
probe_capture_dir = (
    RUNTIME_DIR
    / "instrumentation_probe_events"
)

if probe_capture_dir.exists():
    shutil.rmtree(
        probe_capture_dir
    )

probe_capture_dir.mkdir(
    parents=True,
    exist_ok=True,
)

instrument_probe_env = (
    legacy_runtime_env.copy()
)

instrument_probe_env[
    "LR_CICERO_CAPTURE_DIR"
] = str(
    probe_capture_dir
)

instrument_probe_env[
    "PYTHONPATH"
] = os.pathsep.join(
    [
        str(
            INSTRUMENT_DIR
        ),
        str(
            CICERO_REPO
        ),
    ]
)

instrument_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,sitecustomize; "
            "from fairdiplomacy.agents.bqre1p_agent import BQRE1PAgent; "
            "print(json.dumps({"
            "'sitecustomize':sitecustomize.__file__,"
            "'run_search_module':BQRE1PAgent.run_search.__module__,"
            "'run_search_name':BQRE1PAgent.run_search.__name__,"
            "'bilateral_module':"
            "BQRE1PAgent.run_bilateral_search_with_conditional_evs.__module__,"
            "'bilateral_name':"
            "BQRE1PAgent.run_bilateral_search_with_conditional_evs.__name__"
            "}))"
        ),
    ],
    cwd=CICERO_REPO,
    env=instrument_probe_env,
)

(
    RUNTIME_DIR
    / "instrumentation_probe.txt"
).write_text(
    instrument_probe[
        "output"
    ]
)

print(
    instrument_probe[
        "output"
    ]
)

if instrument_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Planner instrumentation could not be loaded in the isolated runtime."
    )

probe_event_files = list(
    probe_capture_dir.glob(
        "*.jsonl"
    )
)

if not probe_event_files:
    raise RuntimeError(
        "Instrumentation loaded but emitted no probe event."
    )


{"sitecustomize": "/workspace/latent-reservations/notebook_outputs/15_cicero_planner_capture_mechanistic_positive_control/20260815T140016Z/data/instrumentation/sitecustomize.py", "run_search_module": "sitecustomize", "run_search_name": "_lr_capture_run_search", "bilateral_module": "sitecustomize", "bilateral_name": "_lr_capture_run_bilateral_search_with_conditional_evs"}



In [8]:
if CAPTURE_DIR.exists():
    for path in CAPTURE_DIR.glob(
        "*.jsonl"
    ):
        path.unlink()

FULL_AGENT_LOG = (
    RUNTIME_DIR
    / "full_agent_capture.log"
)

compare_command = [
    str(
        LEGACY_PYTHON
    ),
    "run.py",
    "--adhoc",
    "--cfg",
    str(
        COMPARE_CONFIG.relative_to(
            CICERO_REPO
        )
    ),
    f"Iagent_one={CICERO_AGENT_INCLUDE}",
    "use_shared_agent=1",
    "power_one=TURKEY",
]

process_env = (
    legacy_runtime_env.copy()
)

process_env[
    "PYTHONUNBUFFERED"
] = "1"

process_env[
    "LR_CICERO_CAPTURE_DIR"
] = str(
    CAPTURE_DIR
)

process_env[
    "PYTHONPATH"
] = os.pathsep.join(
    [
        str(
            INSTRUMENT_DIR
        ),
        str(
            CICERO_REPO
        ),
    ]
)

process = subprocess.Popen(
    compare_command,
    cwd=CICERO_REPO,
    env=process_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    start_new_session=True,
)

line_queue = queue.Queue()

def reader_thread():
    assert process.stdout is not None

    for line in process.stdout:
        line_queue.put(
            line
        )

    line_queue.put(
        None
    )

thread = threading.Thread(
    target=reader_thread,
    daemon=True,
)

thread.start()

capture_complete = False
process_exited = False
capture_status = {
    "complete": False,
    "planner_event": None,
    "outward_event": None,
}

runtime_samples = []
started = time.monotonic()
last_runtime_sample = 0.0
last_display = started

with FULL_AGENT_LOG.open(
    "w",
    encoding="utf-8",
) as log_handle:
    progress = tqdm(
        desc="Capture CICERO planner event",
        unit="log line",
        mininterval=2.0,
        miniters=100,
        dynamic_ncols=False,
    )

    while True:
        try:
            item = line_queue.get(
                timeout=1.0
            )
        except queue.Empty:
            item = "__NO_LINE__"

        if item is None:
            process_exited = True
            break

        if item != "__NO_LINE__":
            log_handle.write(
                item
            )

            log_handle.flush()

            progress.update(
                1
            )

        now = time.monotonic()

        elapsed = int(
            now
            - started
        )

        progress.set_postfix_str(
            (
                f"running {elapsed // 60:02d}:"
                f"{elapsed % 60:02d}"
            ),
            refresh=False,
        )

        if (
            now
            - last_display
            >= 10.0
        ):
            progress.refresh()
            last_display = now

        events = load_capture_events()

        capture_status = (
            find_capture_completion(
                events
            )
        )

        if capture_status[
            "complete"
        ]:
            capture_complete = True
            break

        if (
            now
            - last_runtime_sample
            >= 15.0
        ):
            sample = capture_command(
                [
                    "nvidia-smi",
                    "--query-compute-apps=pid,process_name,used_memory",
                    "--format=csv,noheader,nounits",
                ]
            )

            runtime_samples.append({
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "output": sample[
                    "output"
                ],
            })

            last_runtime_sample = now

        if process.poll() is not None:
            process_exited = True
            break

        if (
            now
            - started
            > CAPTURE_TIMEOUT_SECONDS
        ):
            break

    progress.close()

if process.poll() is None:
    try:
        os.killpg(
            process.pid,
            signal.SIGINT,
        )

        process.wait(
            timeout=30
        )

    except Exception:
        try:
            os.killpg(
                process.pid,
                signal.SIGTERM,
            )

            process.wait(
                timeout=15
            )

        except Exception:
            os.killpg(
                process.pid,
                signal.SIGKILL,
            )

            process.wait()

(
    RUNTIME_DIR
    / "gpu_runtime_samples.json"
).write_text(
    json.dumps(
        runtime_samples,
        indent=2,
    )
)

launch_result = {
    "command": compare_command,
    "capture_complete": capture_complete,
    "process_exited_before_capture": bool(
        process_exited
        and not capture_complete
    ),
    "returncode_after_stop": process.returncode,
    "event_files": [
        str(
            path
        )
        for path in sorted(
            CAPTURE_DIR.glob(
                "*.jsonl"
            )
        )
    ],
    "log_path": str(
        FULL_AGENT_LOG
    ),
}

(
    TABLE_DIR
    / "launch_result.json"
).write_text(
    json.dumps(
        launch_result,
        indent=2,
    )
)

print(
    json.dumps(
        launch_result,
        indent=2,
    )
)

if not capture_complete:
    events = load_capture_events()

    event_counts = {}

    for event in events:
        event_name = str(
            event.get(
                "event"
            )
        )

        event_counts[
            event_name
        ] = (
            event_counts.get(
                event_name,
                0,
            )
            + 1
        )

    failure_summary = {
        "event_counts": event_counts,
        "recent_events": events[-20:],
        "full_agent_log": str(
            FULL_AGENT_LOG
        ),
        "capture_dir": str(
            CAPTURE_DIR
        ),
    }

    failure_summary_path = (
        TABLE_DIR
        / "capture_failure_summary.json"
    )

    failure_summary_path.write_text(
        json.dumps(
            failure_summary,
            indent=2,
        )
    )

    raise RuntimeError(
        "Planner/outward capture target was not reached. "
        f"Event counts: {event_counts}. "
        f"Inspect {failure_summary_path}."
    )


Capture CICERO planner event: 0log line [00:00, ?log line/s]

{
  "command": [
    "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python",
    "run.py",
    "--adhoc",
    "--cfg",
    "conf/c01_ag_cmp/cmp.prototxt",
    "Iagent_one=agents/cicero.prototxt",
    "use_shared_agent=1",
    "power_one=TURKEY"
  ],
  "capture_complete": true,
  "process_exited_before_capture": false,
  "returncode_after_stop": 1,
  "event_files": [
    "/workspace/latent-reservations/notebook_outputs/15_cicero_planner_capture_mechanistic_positive_control/20260815T140016Z/data/planner_capture_events/205556.jsonl"
  ],
  "log_path": "/workspace/latent-reservations/notebook_outputs/15_cicero_planner_capture_mechanistic_positive_control/20260815T140016Z/data/runtime/full_agent_capture.log"
}


In [9]:
events = load_capture_events()

(
    RUNTIME_DIR
    / "capture_events_merged.json"
).write_text(
    json.dumps(
        events,
        indent=2,
    )
)

completion = find_capture_completion(
    events
)

planner_event = completion[
    "planner_event"
]

outward_event = completion[
    "outward_event"
]

if planner_event is None:
    raise RuntimeError(
        "No completed planner event was captured."
    )

planner_payload = planner_event.get(
    "payload",
    {}
)

result_payload = planner_payload.get(
    "result",
    {}
)

method_outputs = result_payload.get(
    "method_outputs",
    {}
)

utility_outputs = result_payload.get(
    "utility_outputs",
    {}
)

planner_method = planner_payload.get(
    "planner_method"
)

planner_capture = {
    "run_id": RUN_ID,
    "planner_method": planner_method,
    "planner_event": planner_event,
    "outward_event": outward_event,
    "method_outputs": method_outputs,
    "utility_outputs": utility_outputs,
}

(
    TABLE_DIR
    / "planner_capture.json"
).write_text(
    json.dumps(
        planner_capture,
        indent=2,
    )
)

print({
    "captured_event_count": len(
        events
    ),
    "planner_event_type": planner_event.get(
        "event"
    ),
    "planner_method": planner_method,
    "planner_result_type": result_payload.get(
        "type"
    ),
    "policy_methods": sorted(
        method_outputs.keys()
    ),
    "outward_event_type": (
        outward_event.get(
            "event"
        )
        if outward_event is not None
        else None
    ),
})


{'captured_event_count': 14, 'planner_event_type': 'run_bilateral_search_with_conditional_evs_result', 'planner_method': 'run_bilateral_search_with_conditional_evs', 'planner_result_type': 'fairdiplomacy.agents.bqre1p_agent.BRMResult', 'policy_methods': ['get_agent_policy', 'get_bp_policy', 'get_population_policy', 'is_early_exit'], 'outward_event_type': 'generate_message_result'}


In [10]:
def contains_nonempty_payload(
    value,
):
    if value is None:
        return False

    if isinstance(
        value,
        dict,
    ):
        if value.get(
            "__not_called__"
        ):
            return False

        if value.get(
            "__error__"
        ):
            return False

        return any(
            contains_nonempty_payload(
                item
            )
            for item in value.values()
        )

    if isinstance(
        value,
        list,
    ):
        return any(
            contains_nonempty_payload(
                item
            )
            for item in value
        )

    if isinstance(
        value,
        str,
    ):
        return bool(
            value.strip()
        )

    return True


policy_presence = {
    name: contains_nonempty_payload(
        method_outputs.get(
            name
        )
    )
    for name in [
        "get_agent_policy",
        "get_bp_policy",
        "get_population_policy",
    ]
}

planner_text = json.dumps(
    planner_event,
    sort_keys=True,
).lower()

state_surface_presence = {
    "belief": "belief" in planner_text,
    "plausible_orders": (
        "plausible"
        in planner_text
        and "order"
        in planner_text
    ),
    "utility_or_value": (
        "utility"
        in planner_text
        or "value"
        in planner_text
    ),
    "player_type": (
        "ptype"
        in planner_text
        or "player_type"
        in planner_text
    ),
}

mechanistic_target_candidates = {
    "planner_method": planner_method,
    "planner_result_type": result_payload.get(
        "type"
    ),
    "policy_presence": policy_presence,
    "state_surface_presence": state_surface_presence,
    "policy_payloads": {
        name: method_outputs.get(
            name
        )
        for name, present in policy_presence.items()
        if present
    },
    "utility_outputs": utility_outputs,
    "beliefs": result_payload.get(
        "beliefs"
    ),
    "outward_event": outward_event,
}

(
    TABLE_DIR
    / "mechanistic_target_candidates.json"
).write_text(
    json.dumps(
        mechanistic_target_candidates,
        indent=2,
    )
)

print(
    json.dumps(
        {
            "policy_presence": policy_presence,
            "state_surface_presence": state_surface_presence,
            "outward_event_type": (
                outward_event.get(
                    "event"
                )
                if outward_event
                is not None
                else None
            ),
        },
        indent=2,
    )
)


{
  "policy_presence": {
    "get_agent_policy": true,
    "get_bp_policy": true,
    "get_population_policy": true
  },
  "state_surface_presence": {
    "belief": true,
    "plausible_orders": true,
    "utility_or_value": true,
    "player_type": true
  },
  "outward_event_type": "generate_message_result"
}


In [11]:
capture_errors = [
    event
    for event in events
    if str(
        event.get(
            "event",
            "",
        )
    ).endswith(
        "_error"
    )
]

readiness = {
    "frozen_cicero_commit": (
        actual_commit_value
        == CICERO_COMMIT
    ),
    "notebook14_runtime_stack_pass": (
        preflight_probe[
            "returncode"
        ]
        == 0
    ),
    "instrumentation_probe_pass": (
        instrument_probe[
            "returncode"
        ]
        == 0
    ),
    "completed_native_planner_result_captured": (
        planner_event
        is not None
    ),
    "planner_method_identified": (
        planner_method
        in {
            "run_search",
            "run_bilateral_search_with_conditional_evs",
        }
    ),
    "planner_result_type_present": bool(
        result_payload.get(
            "type"
        )
    ),
    "planner_policy_payload_present": any(
        policy_presence.values()
    ),
    "outward_event_paired": (
        outward_event
        is not None
    ),
    "capture_error_events_absent": (
        len(
            capture_errors
        )
        == 0
    ),
}

readiness[
    "ready_for_following_measurement_stage"
] = bool(
    all(
        readiness.values()
    )
)

(
    TABLE_DIR
    / "readiness.json"
).write_text(
    json.dumps(
        readiness,
        indent=2,
    )
)

print(
    json.dumps(
        readiness,
        indent=2,
    )
)

if not readiness[
    "ready_for_following_measurement_stage"
]:
    raise RuntimeError(
        "Planner capture completed incompletely; inspect the readiness artifact."
    )


{
  "frozen_cicero_commit": true,
  "notebook14_runtime_stack_pass": true,
  "instrumentation_probe_pass": true,
  "completed_native_planner_result_captured": true,
  "planner_method_identified": true,
  "planner_result_type_present": true,
  "planner_policy_payload_present": true,
  "outward_event_paired": true,
  "capture_error_events_absent": true,
  "ready_for_following_measurement_stage": true
}


In [12]:
result_summary = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "cicero_commit": CICERO_COMMIT,
    "purpose": (
        "Capture a completed released CICERO native planner result "
        "and pair it with subsequent outward behavior."
    ),
    "scientific_status": (
        "Mechanistic capture/serialization positive control only; "
        "no activation-vs-text H1 estimate."
    ),
    "preflight": preflight,
    "runtime_signatures": runtime_signatures,
    "launch_result": launch_result,
    "planner_method": planner_method,
    "planner_result_type": result_payload.get(
        "type"
    ),
    "policy_presence": policy_presence,
    "state_surface_presence": state_surface_presence,
    "outward_event_type": (
        outward_event.get(
            "event"
        )
        if outward_event is not None
        else None
    ),
    "readiness": readiness,
}

(
    TABLE_DIR
    / "result_summary.json"
).write_text(
    json.dumps(
        result_summary,
        indent=2,
    )
)

manifest = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "run_output_dir": str(
        RUN_DIR
    ),
    "preflight": str(
        TABLE_DIR
        / "preflight.json"
    ),
    "runtime_signatures": str(
        TABLE_DIR
        / "runtime_signatures.json"
    ),
    "instrumentation_source": str(
        SITE_CUSTOMIZE
    ),
    "instrumentation_probe": str(
        RUNTIME_DIR
        / "instrumentation_probe.txt"
    ),
    "full_agent_log": str(
        FULL_AGENT_LOG
    ),
    "capture_event_directory": str(
        CAPTURE_DIR
    ),
    "capture_events_merged": str(
        RUNTIME_DIR
        / "capture_events_merged.json"
    ),
    "planner_capture": str(
        TABLE_DIR
        / "planner_capture.json"
    ),
    "mechanistic_target_candidates": str(
        TABLE_DIR
        / "mechanistic_target_candidates.json"
    ),
    "readiness": str(
        TABLE_DIR
        / "readiness.json"
    ),
    "result_summary": str(
        TABLE_DIR
        / "result_summary.json"
    ),
}

(
    MANIFEST_DIR
    / "notebook15_manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

print(
    json.dumps(
        result_summary,
        indent=2,
    )
)

print()
print(
    "Notebook 15 complete."
)
print(
    "Result summary:",
    TABLE_DIR
    / "result_summary.json",
)
print(
    "Run directory:",
    RUN_DIR,
)


{
  "notebook_build": "latent-reservations-notebook15-cicero-planner-capture-v2",
  "run_id": "20260815T140016Z",
  "cicero_commit": "e85afeddb34f5b7c1ea0827203b425a0f7e68ead",
  "purpose": "Capture a completed released CICERO native planner result and pair it with subsequent outward behavior.",
  "scientific_status": "Mechanistic capture/serialization positive control only; no activation-vs-text H1 estimate.",
  "preflight": {
    "cicero_commit": "e85afeddb34f5b7c1ea0827203b425a0f7e68ead",
    "stack_markers": [
      "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/.latent_reservations_cicero_stack_v5_7"
    ],
    "torch_route": {
      "version": "1.7.1+cu110",
      "cuda": "11.0",
      "cuda_available": true,
      "root": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/lib/python3.7/site-packages/torch",
      "lib": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/lib/python3.7/site-packages/torch/lib"
    